# 11 — Day-30 Temporal Check — Training on Older Cases, Testing on Newer Cases

**Author:** Ali Čikić · General Assembly – Data Science Bootcamp

## 1. Goal

Notebook 10 found a modest day-30 signal for Logistic Regression using a **random** train/test split. In practice a model
would be trained on **older** cases and used on **newer** ones. This notebook checks whether the same fixed day-30 model
still separates long and not-long cases when it is trained on older cases and tested on newer cases.

This is a separate **temporal robustness check**. For this notebook only, it replaces the random split.

Everything else stays the same as notebook 10:

- **Completed cases:** `case:endDate` is present.
- **Recorded activity span:** last event minus first event (not necessarily the administrative completion time).
- **Eligible:** last event after first event + 30 days (a retrospective rule, as explained in notebooks 07 and 10).
- **Target:** `long_case = 1` if the recorded activity span is more than 63 days.
- The same seven count features, the same activity rule, `StandardScaler` + `LogisticRegression(max_iter=1000)` with the
  default threshold, and a majority baseline.

## 2. Load the Data and Build the Day-30 Cohort

In [1]:
import os
import warnings
from pathlib import Path

os.environ["PM4PY_SHOW_PROGRESS_BAR"] = "false"  # no progress bar while loading
# hide only PM4Py's optional speed-up tip ("Install the optional requirement `r4pm` ..."); other warnings still show
warnings.filterwarnings("ignore", message="Install the optional requirement `r4pm`")

import pandas as pd
import pm4py
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# the project folder: works whether Jupyter starts in the notebooks folder or in the project folder
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_path = project_root / "data" / "raw" / "BPIC15_1.xes"



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




*When PM4Py is imported it prints a short welcome and licence message. This is normal and not an error.*

In [2]:
log = pm4py.read_xes(str(data_path))
df = pm4py.convert_to_dataframe(log)

case_data = df.groupby("case:concept:name").agg(
    first_event=("time:timestamp", "min"),
    last_event=("time:timestamp", "max"),
    end_date=("case:endDate", "first"),
)
case_data["duration_days"] = (case_data["last_event"] - case_data["first_event"]).dt.total_seconds() / (24 * 60 * 60)
case_data = case_data[case_data["end_date"].notna()]   # completed cases only
case_data["long_case"] = (case_data["duration_days"] > 63).astype(int)

case_data["prediction_time"] = case_data["first_event"] + pd.Timedelta(days=30)
active = case_data[case_data["last_event"] > case_data["prediction_time"]]

print("Completed cases:", len(case_data))
print("Day-30 eligible cases:", len(active))

Completed cases: 956
Day-30 eligible cases: 838


## 3. Chronological Split

1. Sort the eligible cases by `prediction_time` (first event + 30 days).
2. The **boundary** is the `prediction_time` of the case at the start of the final 20% of this sorted list.
3. **Test group:** all cases with `prediction_time` at or after the boundary (cases with the same timestamp stay together).
4. **Candidate training cases:** `prediction_time` before the boundary.
5. **Training group:** candidates whose `last_event` **and** `case:endDate` are both before the boundary. Older cases whose
   recorded outcome was still unresolved at the boundary are left out, because at that moment we would not yet know their outcome.

The boundary is fixed by this rule and is not changed after seeing the sizes, classes or results. Note that even this
rule cannot prove **when** information was actually entered into the source system; it relies on the recorded dates.

In [3]:
sorted_cases = active.sort_values("prediction_time", kind="stable")
boundary_position = int(len(sorted_cases) * 0.8)
boundary = sorted_cases["prediction_time"].iloc[boundary_position]

test_cases = sorted_cases[sorted_cases["prediction_time"] >= boundary]
candidates = sorted_cases[sorted_cases["prediction_time"] < boundary]
train_cases = candidates[(candidates["last_event"] < boundary) & (candidates["end_date"] < boundary)]

print("Boundary:", boundary)
print("Test cases:", len(test_cases))
print("Candidate training cases:", len(candidates))
print("Training cases after the date rule:", len(train_cases))
print("Candidates excluded by the date rule:", len(candidates) - len(train_cases),
      "| of which long:", candidates.drop(train_cases.index)["long_case"].sum())

Boundary: 2013-10-25 22:00:00+00:00
Test cases: 168
Candidate training cases: 670
Training cases after the date rule: 617
Candidates excluded by the date rule: 53 | of which long: 44


### Feasibility check (before any modelling)

Practical minimums for this task: both groups must contain both classes, training needs at least 100 cases and test at least
30 cases. These are not guarantees of statistical reliability.

In [4]:
feasibility = pd.DataFrame({
    "cases": [len(train_cases), len(test_cases)],
    "long": [train_cases["long_case"].sum(), test_cases["long_case"].sum()],
    "not long": [(train_cases["long_case"] == 0).sum(), (test_cases["long_case"] == 0).sum()],
    "first prediction_time": [train_cases["prediction_time"].min(), test_cases["prediction_time"].min()],
    "last prediction_time": [train_cases["prediction_time"].max(), test_cases["prediction_time"].max()],
}, index=["training", "test"])
feasibility["share long"] = (feasibility["long"] / feasibility["cases"]).round(3)
display(feasibility)

feasible = (
    len(train_cases) >= 100 and len(test_cases) >= 30
    and train_cases["long_case"].nunique() == 2 and test_cases["long_case"].nunique() == 2
)
print("Feasible:", feasible)
assert feasible, "Predefined feasibility check failed: stop here and do not model."

,cases,long,not long,first prediction_time,last prediction_time,share long
training,617,328,289,2010-11-04 22:00:00+00:00,2013-10-02 22:00:00+00:00,0.532
test,168,100,68,2013-10-25 22:00:00+00:00,2015-02-13 23:00:00+00:00,0.595


Feasible: True


In [5]:
print("Overlap between training and test:", len(set(train_cases.index) & set(test_cases.index)))
print("All training last events before the boundary:", (train_cases["last_event"] < boundary).all())
print("All training end dates before the boundary:", (train_cases["end_date"] < boundary).all())
print("All test cases at or after the boundary:", (test_cases["prediction_time"] >= boundary).all())

Overlap between training and test: 0
All training last events before the boundary: True
All training end dates before the boundary: True
All test cases at or after the boundary: True


### Overlap with the earlier random split

The temporal groups come from the same 838 day-30 cases as notebook 10, so they overlap with the random split used in
notebooks 07–10. I rebuild that random split (same code as notebook 10) and count the overlap.

In [6]:
day14_cases = case_data[case_data["last_event"] > case_data["first_event"] + pd.Timedelta(days=14)]
random_train_ids, random_test_ids = train_test_split(
    day14_cases.index, test_size=0.20, random_state=42, stratify=day14_cases["long_case"]
)

print("Temporal test cases that were random-split TRAINING cases:", len(set(test_cases.index) & set(random_train_ids)))
print("Temporal test cases that were random-split TEST cases:    ", len(set(test_cases.index) & set(random_test_ids)))
print("Temporal training cases that were random-split TEST cases:", len(set(train_cases.index) & set(random_test_ids)))

Temporal test cases that were random-split TRAINING cases: 138
Temporal test cases that were random-split TEST cases:     30
Temporal training cases that were random-split TEST cases: 121


So the temporal test is **not a fresh holdout**: most of its cases were used to train and compare models in notebooks
07–10, and the features, settings and threshold were chosen while looking at those notebooks. It checks whether the same
design still works when the model is trained only on older cases; it is not an independent final test.

## 4. Day-30 Features (Same as Notebook 10)

Only events timestamped **on or before** each case's own day-30 cutoff are used, for training and test cases alike.

In [7]:
def events_by_day30(event_table, cases):
    # keep only events of the given cases that happened on or before that case's day-30 cutoff
    events = event_table[event_table["case:concept:name"].isin(cases.index)].copy()
    events["prediction_time"] = events["case:concept:name"].map(cases["prediction_time"])
    events["first_event"] = events["case:concept:name"].map(cases["first_event"])
    first30 = events[events["time:timestamp"] <= events["prediction_time"]].copy()
    first30["day"] = (first30["time:timestamp"] - first30["first_event"]).dt.total_seconds() / (24 * 60 * 60)
    return first30

**The seven count features.** `count_columns` fixes their order; `count_features` calculates them for every case from the events kept above.

In [8]:
count_columns = [
    "events_first30",
    "unique_activities_first30",
    "unique_resources_first30",
    "repeated_activities_first30",
    "events_first15",
    "events_days16_30",
    "days_since_last_event",
]


def count_features(first30, cases):
    first30 = first30.copy()
    first30["in_first15"] = first30["day"] <= 15
    features = first30.groupby("case:concept:name").agg(
        events_first30=("concept:name", "size"),
        unique_activities_first30=("concept:name", "nunique"),
        unique_resources_first30=("org:resource", "nunique"),
        events_first15=("in_first15", "sum"),
        last_event_by_day30=("time:timestamp", "max"),
    )
    features["repeated_activities_first30"] = features["events_first30"] - features["unique_activities_first30"]
    features["events_days16_30"] = features["events_first30"] - features["events_first15"]
    features["days_since_last_event"] = (
        cases.loc[features.index, "prediction_time"] - features["last_event_by_day30"]
    ).dt.total_seconds() / (24 * 60 * 60)
    return features[count_columns]

**Apply the helpers** to the cases and check that no event after the cutoff was used. Training and test cases each use their own cutoff.

In [9]:
modelling_cases = pd.concat([train_cases, test_cases])
first30 = events_by_day30(df, modelling_cases)
counts = count_features(first30, modelling_cases)

print("Latest event used, in days after the case start:", first30["day"].max())
print("Every event used is on or before its case's cutoff:", (first30["time:timestamp"] <= first30["prediction_time"]).all())
print("Cases with features:", len(counts))

Latest event used, in days after the case start: 30.0
Every event used is on or before its case's cutoff: True
Cases with features: 785


### Activity indicators (selected on training cases only)

Activity codes recorded by day 30 in **10–90%** of the **training** cases, at most the **15 most frequent**, ties broken by
activity code. The target is not used. A 0 means "not recorded by day 30".

**Choose activity codes using training cases only.** `select_activities` returns the codes recorded in 10–90% of the given cases (at most 15, most frequent first, ties broken by code). It is only ever called with training cases.

In [10]:
def select_activities(first30, case_ids):
    # activity codes recorded in 10-90% of the given cases, at most 15, most frequent first (ties by code)
    events = first30[first30["case:concept:name"].isin(case_ids)]
    share = events.groupby("concept:name")["case:concept:name"].nunique() / len(case_ids) * 100
    share = share.rename("train_cases_pct").reset_index()
    share = share[(share["train_cases_pct"] >= 10) & (share["train_cases_pct"] <= 90)]
    share = share.sort_values(["train_cases_pct", "concept:name"], ascending=[False, True])
    return share.head(15)

**Turn the chosen activities into 0/1 columns.** `activity_flags` gives 1 if the activity was recorded by the cutoff in that case, otherwise 0.

In [11]:
def activity_flags(first30, activities, case_ids):
    # 1 if the activity was recorded by day 30 in that case, otherwise 0
    flags = pd.DataFrame(index=case_ids)
    for activity in activities:
        cases_with_activity = first30.loc[first30["concept:name"] == activity, "case:concept:name"].unique()
        flags["act_" + activity] = flags.index.isin(cases_with_activity).astype(int)
    return flags

**Select the activities on the older training cases** and show them with the English label recorded most often.

In [12]:
train_ids = train_cases.index
test_ids = test_cases.index

selected = select_activities(first30, train_ids)
train_events = first30[first30["case:concept:name"].isin(train_ids)]

# the English label recorded most often for each selected activity code
english_names = []
for activity in selected["concept:name"]:
    labels = train_events.loc[train_events["concept:name"] == activity, "activityNameEN"]
    english_names.append(labels.mode()[0])

selected_table = selected[["concept:name"]].copy()
selected_table["activityNameEN"] = english_names
selected_table["train_cases_pct"] = selected["train_cases_pct"]
selected_table.round(1).reset_index(drop=True)

,concept:name,activityNameEN,train_cases_pct
0,01_HOOFD_030_2,enter senddate acknowledgement,89.5
1,01_HOOFD_030_1,send confirmation receipt,89.3
2,01_HOOFD_065_2,enter senddate procedure confirmation,66.8
3,01_HOOFD_101,registration date publication,63.7
4,01_HOOFD_065_1,send procedure confirmation,59.2
5,01_HOOFD_011,OLO messaging active,53.8
6,02_DRZ_010,forward to the competent authority,53.6
7,04_BPT_005,regular procedure without MER,53.0
8,01_HOOFD_061,start WABOprocedure,35.2
9,01_HOOFD_040,forward to the competent authority,32.7


## 5. Fit on Older Cases, Score Once on Newer Cases

The majority baseline and the Logistic Regression pipeline are fitted on the **training** cases only (the scaler is part of
the pipeline). Each approach is scored **once** on the newer test cases. No cross-validation or tuning.

In [13]:
activities = selected["concept:name"].tolist()
X_train = counts.loc[train_ids].join(activity_flags(first30, activities, train_ids))
X_test = counts.loc[test_ids].join(activity_flags(first30, activities, test_ids))
y_train = train_cases["long_case"]
y_test = test_cases["long_case"]

**Fit the two approaches on the older cases only.**

In [14]:
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)

**Score each approach once on the newer test cases.**

In [15]:
results = {}
for name, model in [("Majority baseline", baseline), ("Logistic Regression", logreg)]:
    predictions = model.predict(X_test)
    results[name] = {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions),
        "ROC-AUC": roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]),
        "Cases flagged long": int(predictions.sum()),
    }

print("Test cases:", len(y_test), f"| share of long cases in test: {y_test.mean():.1%}",
      "| majority class in training:", y_train.mode()[0])
results_table = pd.DataFrame(results).T
results_table["Cases flagged long"] = results_table["Cases flagged long"].astype(int)
results_table.round(3)

Test cases: 168 | share of long cases in test: 59.5% | majority class in training: 1


,Accuracy,Precision,Recall,F1,ROC-AUC,Cases flagged long
Majority baseline,0.595,0.595,1.00,0.746,0.500,168
Logistic Regression,0.637,0.775,0.55,0.643,0.678,71


In [16]:
pd.DataFrame(
    confusion_matrix(y_test, logreg.predict(X_test)),
    index=["actual 0 (not long)", "actual 1 (long)"],
    columns=["predicted 0", "predicted 1"],
)

,predicted 0,predicted 1
actual 0 (not long),52,16
actual 1 (long),45,55


## 6. Interpretation

**Compared with its own baseline.** On the 168 newer test cases, Logistic Regression ranks long cases above not-long cases
better than chance (ROC-AUC 0.678 vs 0.500). Its warnings are more often correct than random flagging: 77.5% of the 71
flagged cases are long, compared with 59.5% long cases in the test group. But it finds only 55% of the long cases (55 of 100)
and misses 45. The baseline flags every case, so it has higher recall (1.00) and F1 (0.746 vs 0.643), but it cannot tell
cases apart at all.

**Does this support notebook 10?** Mostly yes, for **ranking**: the ROC-AUC on newer cases (0.678) is in the same range as
notebook 10 (0.702 in cross-validation, 0.659 on its random test cases). The model's **warning behaviour** is different: it
flags fewer cases, so precision is higher and recall is lower than in notebook 10 (recall 0.716 there).

**Why the cohorts are not directly comparable.** This test group contains the newest cases, and its share of long cases
(59.5%) is higher than in the training group (53.2%). The date rule removed 53 older candidates whose outcome was not yet
resolved at the boundary, and 44 of them were long cases, so the training group has fewer long cases than it otherwise would.
Changes in the process over time may also play a role. Accuracy should therefore only be compared with this test group's
own baseline (59.5%), not with notebook 10's accuracy.

**Conclusion.** Training on older cases and testing on newer cases gives a similar, modest ranking signal at day 30. This
weakens the worry that notebook 10's result depended only on a lucky random split, but it is not independent evidence:
138 of the 168 test cases were training cases in notebooks 07–10. It does not show practical usefulness:
the model still misses almost half of the long cases at the default threshold.

## 7. Limitations

- This is an **exploratory** temporal check, not an untouched holdout: many of these cases were used in earlier notebooks.
- Only **historically completed** cases (with `case:endDate`) are included.
- The target is the **recorded activity span**, which may include publication, objection, archiving or later recording steps.
- The date rule relies on recorded dates; it cannot prove when information was actually entered into the source system.
- One Dutch municipality; not SPAR or Montenegro data.